Preparar modelo BERTopic sobre todos los chunks
------------------------------------------------
- Usa todos los chunks del corpus
- Aplica reduce_outliers y update_topics
- Guarda modelo y resultados
- Permite recargar sin recalcular si ya existe

In [18]:
import os, pickle, json
from pathlib import Path
import pandas as pd
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction import text
import spacy


# Cargar Datos


In [74]:
DATA_PATH = Path("../data/processed/chunks.parquet")
MODEL_DIR = Path("../models")
#MODEL_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = MODEL_DIR / "topicos_chunks"
TOPICS_PATH = MODEL_DIR / "topicos_chunks_info.parquet"
EMBEDDINGS_PATH = MODEL_DIR / "embeddings_chunks.pkl"
EMBEDDER_PATH = MODEL_DIR / "embedder_model.pkl"

# Función bertopic

In [16]:
df = pd.read_parquet(DATA_PATH)

df.head()

,id_doc,autor_doc,fecha_doc,diario_doc,titulo_doc,chunk_id,texto_chunk
0,1,Gonzalo Hernández,2018-01-01,El Espectador,Fajardo: para nada tibio,0,"La Coalición Colombia Partido Alianza Verde, P..."
1,1,Gonzalo Hernández,2018-01-01,El Espectador,Fajardo: para nada tibio,1,posible costo electoral que pocos asumen. Mani...
2,2,Eduardo Barajas Sandoval,2018-01-01,El Espectador,Macedonia de Norte,0,Las interpretaciones de la historia sirven com...
3,2,Eduardo Barajas Sandoval,2018-01-01,El Espectador,Macedonia de Norte,1,"y culturales, y del poder político: la hoy Mac..."
4,2,Eduardo Barajas Sandoval,2018-01-01,El Espectador,Macedonia de Norte,2,"Olimpo. En gracia de discusión, y solamente co..."


In [75]:
# ===========================
# FUNCIÓN PRINCIPAL
# ===========================
def preparar_bertopic(recargar: bool = True):
    """
    Entrena o carga un modelo BERTopic sobre todos los chunks ya limpios.

    Parámetros
    ----------
    recargar : bool
        Si True, recalcula el modelo y guarda resultados.
        Si False, carga modelo y resultados previos.

    Retorna
    -------
    model : BERTopic
    topics_df : pd.DataFrame
    df_chunks : pd.DataFrame
    """

    # ===========================
    # 1. Cargar modelo si ya existe
    # ===========================
    if not recargar and MODEL_PATH.exists() and TOPICS_PATH.exists():
        print("🔹 Cargando modelo, embeddings y tópicos guardados...")
        model = BERTopic.load(MODEL_PATH)
        df_chunks = pd.read_parquet(DATA_PATH)
        topics_df = pd.read_parquet(TOPICS_PATH)

        # Reasignar embedder si está guardado
        if EMBEDDER_PATH.exists():
            with open(EMBEDDER_PATH, "rb") as f:
                embedder = pickle.load(f)
            model.embedding_model = embedder
            print("✅ Embedding model re-asignado correctamente.")
        else:
            print("⚠️ No se encontró EMBEDDER_PATH. Se cargará al usar transform().")

        print(f"✅ Modelo cargado ({len(topics_df)} tópicos).")
        return model, topics_df, df_chunks

    # ===========================
    # 2. Cargar corpus limpio
    # ===========================
    print("📥 Cargando corpus limpio...")
    df_chunks = pd.read_parquet(DATA_PATH)
    textos = df_chunks["texto_chunk"].fillna("").tolist()
    print(f"✅ Total de chunks: {len(textos):,}")

    # ===========================
    # 3. Stopwords de spaCy
    # ===========================
    print("🔤 Cargando stopwords en español (spaCy)...")
    nlp = spacy.blank("es")
    stopwords_es = nlp.Defaults.stop_words
    stopwords_extra = {"colombia", "colombiano", "años", "año", "día", "semana", "mes"}
    stopwords_total = stopwords_es.union(stopwords_extra)

    # ===========================
    # 4. Embeddings rápidos
    # ===========================
    model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

    if not recargar and EMBEDDINGS_PATH.exists() and EMBEDDER_PATH.exists():
        print("🔹 Cargando embeddings y modelo guardado...")
        with open(EMBEDDINGS_PATH, "rb") as f:
            embeddings = pickle.load(f)
        with open(EMBEDDER_PATH, "rb") as f:
            embedder = pickle.load(f)
    else:
        print(f"⚙️ Generando embeddings con {model_name} ...")
        embedder = SentenceTransformer(model_name)
        embeddings = embedder.encode(textos, show_progress_bar=True, batch_size=128)

        with open(EMBEDDINGS_PATH, "wb") as f:
            pickle.dump(embeddings, f)
        with open(EMBEDDER_PATH, "wb") as f:
            pickle.dump(embedder, f)

        print("✅ Embeddings y modelo guardados.")

    # ===========================
    # 5. Vectorizador con stopwords
    # ===========================
    vectorizer_model = CountVectorizer(
        stop_words=list(stopwords_total),
        ngram_range=(1, 2),
        min_df=5
    )

    # ===========================
    # 6. Entrenar BERTopic
    # ===========================
    print("🚀 Entrenando modelo BERTopic...")
    model = BERTopic(
        language="spanish",
        vectorizer_model=vectorizer_model,
        embedding_model=embedder,  # 🔹 Se incrusta el embedder directamente
        calculate_probabilities=False,
        verbose=True
    )

    topics, _ = model.fit_transform(textos, embeddings)

    # ===========================
    # 7. Reducir outliers + actualizar
    # ===========================
    print("🔧 Refinando tópicos...")
    topics = model.reduce_outliers(textos, topics, strategy="c-tf-idf", vectorizer_model=vectorizer_model)
    model.update_topics(textos, topics, vectorizer_model=vectorizer_model, n_gram_range=(1, 2))

    # ===========================
    # 8. Guardar resultados
    # ===========================
    topics_df = model.get_topic_info()
    print(f"📊 Tópicos generados: {len(topics_df)}")

    model.save(MODEL_PATH, serialization="pytorch", save_ctfidf=True)
    topics_df.to_parquet(TOPICS_PATH, index=False)
    print("💾 Modelo y tópicos guardados correctamente.")

    return model, topics_df, df_chunks

# Visualizaciones

In [67]:
def visualizar_topicos(model):
    """
    Devuelve las visualizaciones principales del modelo BERTopic.
    No recalcula nada, solo usa el modelo cargado.
    """
    fig_bar = model.visualize_barchart(top_n_topics=15)
    return fig_bar


def evolucion_temporal(df_chunks, model, year_col="año", text_col="texto_chunk"):
    """
    Calcula la evolución de tópicos por año (3 timestamps).
    Soporta modelos cargados sin embedding_model.
    """
    import matplotlib.pyplot as plt
    from sentence_transformers import SentenceTransformer
    import numpy as np

    print("🕒 Calculando distribución de tópicos por año...")

    # --- 1. Cargar el modelo de embeddings explícitamente ---
    embedding_model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

    # --- 2. Calcular los embeddings directamente ---
    textos = df_chunks[text_col].fillna("").tolist()
    print("🔹 Calculando embeddings temporales...")
    embeddings = embedding_model.encode(textos, show_progress_bar=True, batch_size=128)

    # --- 3. Transformar documentos a tópicos usando los embeddings ---
    print("🔹 Asignando tópicos...")
    topics, _ = model.transform(textos, embeddings=embeddings)
    df_chunks["topic"] = topics

    # --- 4. Agrupar por año ---
    df_yearly = (
        df_chunks.groupby([year_col, "topic"])
        .size()
        .reset_index(name="count")
    )
    df_yearly["proportion"] = (
        df_yearly["count"] / df_yearly.groupby(year_col)["count"].transform("sum")
    )

    # --- 5. Seleccionar 3 años represen




In [76]:
model, topics_df, df_chunks = preparar_bertopic(recargar=True)

📥 Cargando corpus limpio...
✅ Total de chunks: 35,319
🔤 Cargando stopwords en español (spaCy)...
⚙️ Generando embeddings con sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 ...


Batches:   0%|          | 0/276 [00:00<?, ?it/s]

2025-10-16 10:38:17,201 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm


✅ Embeddings y modelo guardados.
🚀 Entrenando modelo BERTopic...


2025-10-16 10:38:28,945 - BERTopic - Dimensionality - Completed ✓
2025-10-16 10:38:28,951 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-10-16 10:38:53,379 - BERTopic - Cluster - Completed ✓
2025-10-16 10:38:53,394 - BERTopic - Representation - Fine-tuning topics using representation models.
2025-10-16 10:39:14,550 - BERTopic - Representation - Completed ✓


🔧 Refinando tópicos...


TypeError: BERTopic.reduce_outliers() got an unexpected keyword argument 'vectorizer_model'

In [69]:
df_chunks['año'] = df_chunks['fecha_doc'].dt.year
df_chunks.head()

,id_doc,autor_doc,fecha_doc,diario_doc,titulo_doc,chunk_id,texto_chunk,año
0,1,Gonzalo Hernández,2018-01-01,El Espectador,Fajardo: para nada tibio,0,"La Coalición Colombia Partido Alianza Verde, P...",2018
1,1,Gonzalo Hernández,2018-01-01,El Espectador,Fajardo: para nada tibio,1,posible costo electoral que pocos asumen. Mani...,2018
2,2,Eduardo Barajas Sandoval,2018-01-01,El Espectador,Macedonia de Norte,0,Las interpretaciones de la historia sirven com...,2018
3,2,Eduardo Barajas Sandoval,2018-01-01,El Espectador,Macedonia de Norte,1,"y culturales, y del poder político: la hoy Mac...",2018
4,2,Eduardo Barajas Sandoval,2018-01-01,El Espectador,Macedonia de Norte,2,"Olimpo. En gracia de discusión, y solamente co...",2018


In [70]:
topics_df

,Topic,Count,Name,Representation,Representative_Docs
0,-1,20719,-1_gobierno_país_paz_política,"[gobierno, país, paz, política, presidente, du...",NaN
1,0,1279,0_salud_virus_coronavirus_pandemia,"[salud, virus, coronavirus, pandemia, covid19,...",NaN
2,1,992,1_educación_estudiantes_universidad_universidades,"[educación, estudiantes, universidad, universi...",NaN
3,2,590,2_mujeres_mujer_hombres_feminismo,"[mujeres, mujer, hombres, feminismo, género, s...",NaN
4,3,552,3_fútbol_jugadores_equipo_equipos,"[fútbol, jugadores, equipo, equipos, copa, jue...",NaN
...,...,...,...,...,...
223,222,10,222_criminal_criminales_política criminal_recl...,"[criminal, criminales, política criminal, recl...",NaN
224,223,10,223_hernández_ñeñe_audios_ñeñe hernández,"[hernández, ñeñe, audios, ñeñe hernández, grab...",NaN
225,224,10,224_misión_liderazgo_razón pública_universidad,"[misión, liderazgo, razón pública, universidad...",NaN
226,225,10,225_especies_pnn_orquídeas_aves,"[especies, pnn, orquídeas, aves, biodiversidad...",NaN


In [72]:
 # === Visualizaciones interactivas ===
fig_bar= visualizar_topicos(model)
fig_bar.show()


In [73]:
evolucion_temporal(df_chunks, model, year_col="año", text_col="texto_chunk")


🕒 Calculando distribución de tópicos por año...
🔹 Calculando embeddings temporales...


Batches:   0%|          | 0/276 [00:00<?, ?it/s]

KeyboardInterrupt: 